<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Political_Violence_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba por Ciudad

In [1]:
import requests
from bs4 import BeautifulSoup

def violence_news_index(city_country):

    terms = [
        "political violence",
        "violent protest",
        "political unrest",
        "riot",
        "clashes with police",
        "political clashes",
        "mob attack politics",
        "political riots",
        "protest violence",
        "extremist attack politics"
    ]

    total = 0

    for term in terms:

        query = f"{term} {city_country}".replace(" ", "+")
        url = f"https://news.google.com/rss/search?q={query}"

        r = requests.get(url)
        soup = BeautifulSoup(r.text, "xml")

        items = soup.find_all("item")

        total += len(items)

    return total

In [30]:
violence_news_index("Buenos Aires Argentina")

758

# Latinobarómetro

In [2]:
from pandas.io.stata import StataReader

with StataReader("https://github.com/LCaravaggio/Happiness_Polarization/raw/refs/heads/main/Latinobarometro_2024_Stata_esp_v20250817.dta") as reader:
    df = reader.read(convert_categoricals=False)
    labels = reader.value_labels()

In [ ]:
import re
from tqdm.auto import tqdm

def clean_city_label(x):

    x = str(x)

    # eliminar prefijo país
    x = re.sub(r"^[A-Z]{2}:\s*", "", x)

    # reemplazos
    x = x.replace("-", " ")
    x = x.replace("/", " ")

    # eliminar cualquier [%...%] si aparece
    x = re.sub(r"\[%.*?%\]", "", x)

    # normalizar espacios
    x = re.sub(r"\s+", " ", x).strip()

    return x

pairs = df[["IDENPA","CIUDAD"]].drop_duplicates()
results = []

for _, row in tqdm(pairs.iterrows(), total=len(pairs)):

    country = labels["IDENPA"].get(row.IDENPA, str(row.IDENPA))
    city = labels["CIUDAD"].get(row.CIUDAD, str(row.CIUDAD))

    city = clean_city_label(city)

    query = f"{city} {country} after:2024-08-22 before:2024-10-10"

    try:
        index = violence_news_index(query)
    except:
        index = None

    results.append({
        "country": country,
        "city": city,
        "query": query,
        "violence_index": index
    })

  0%|          | 0/1062 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
from google.colab import files

violence_df = pd.DataFrame(results)
violence_df.to_csv("violence_news_index.csv", index=False)

files.download("violence_news_index.csv")